# Creates VirtualZarr store from CESM2-WACCM-Historical NetCDFs, then rechunks and writes to Icechunk store on s3.

- run on an m8g.4xlarge w/ 32 workers


In [1]:
import xarray as xr
import zarr
from obstore.store import from_url
import obstore as obs
from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry
from distributed import Client
import icechunk
from icechunk.xarray import to_icechunk
import warnings

warnings.filterwarnings("ignore", module="zarr.*")
warnings.filterwarnings("ignore", module="numcodecs.*")
warnings.filterwarnings("ignore")

zarr.config.set({"async.concurrency": 128})

In [2]:
client = Client(n_workers=16)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,Workers: 16
Total threads: 16,Total memory: 60.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44039,Workers: 0
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34603,Total threads: 1
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/37141/status,Memory: 3.79 GiB
Nanny: tcp://127.0.0.1:40921,


2025-09-18 21:49:25,766 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ffb4911920b5f520614c74fd25407a54 initialized by task ('rechunk-merge-rechunk-transfer-cf87d8827e5c0978a3bdf718211189a8', 0, 0, 0, 9, 0, 0) executed on worker tcp://127.0.0.1:34603
2025-09-18 21:50:07,850 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d990d8e69b3bdf207e0188a97de3d8b8 initialized by task ('rechunk-merge-rechunk-transfer-525b7ad219f77c5cbf8c741e02bfa7d9', 0, 0, 0, 9, 0, 0) executed on worker tcp://127.0.0.1:44773
2025-09-18 21:50:08,833 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ffb4911920b5f520614c74fd25407a54 deactivated due to stimulus 'task-finished-1758232208.8328323'
2025-09-18 21:50:48,886 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle e9dea7ed7a60aaa86eb0e97841700be4 initialized by task ('rechunk-merge-rechunk-transfer-e540b72170fca390412224a0ac71b0d9', 0, 0, 0, 9, 0, 0) executed on worker tcp://127.0.0.1:37653
2025-09-18 21:50:49,472 - di

In [13]:
bucket = "s3://carbonplan-srm/"
prefix = "input/tensor/CESM2-WACCM-Historical/netcdf"
virtual_ic_prefix = "input/tensor/CESM2-WACCM-Historical/icechunk/virtual_icechunk"
ic_prefix = "input/tensor/CESM2-WACCM-Historical/icechunk/icechunk"
store = from_url(bucket, region="us-west-2")
registry = ObjectStoreRegistry({bucket: store})
drop_variables = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]
parser = HDFParser(drop_variables=drop_variables)

In [14]:
stream = obs.list_with_delimiter(store, prefix=prefix, return_arrow=True)
netcdf_list = list(stream["objects"]["path"].to_numpy())
netcdf_list.remove(prefix)
netcdf_urls = [bucket + netcdf_path for netcdf_path in netcdf_list]


def preprocess(ds):
    """
    get ensemble member from ds attrs filename
    """
    ensemble = ds.attrs["case"].rsplit(".")[-1]

    ds = ds.expand_dims({"ensemble_member": [ensemble]})
    return ds

In [15]:
netcdf_urls

['s3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FLDS.19780101-19871231.nc',
 's3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FLDS.19880101-19971231.nc',
 's3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FLDS.19980101-19991231.nc',
 's3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FLDS.19991231-20091231.nc',
 's3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FLDS.20100101-20150115.nc',
 's3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/netcdf/b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h1.FSDS.19780101-19871231.nc',
 's3://carbonplan-srm/

In [16]:
warnings.filterwarnings("ignore", category=UserWarning)

combined_vds = open_virtual_mfdataset(
    netcdf_urls,
    registry=registry,
    parser=parser,
    # preprocess=preprocess,
    combine="by_coords",
    combine_attrs="drop_conflicts",
    loadable_variables=["lat", "lev", "ilev", "time", "nbnd", "lon"],
    parallel="dask",
)
combined_vds

/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:

<xarray.Dataset> Size: 30GB
Dimensions:   (time: 13522, lat: 192, lon: 288, lev: 70, ilev: 71)
Coordinates:
  * lat       (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * lev       (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * ilev      (ilev) float64 568B 4.5e-06 7.42e-06 1.223e-05 ... 985.1 1e+03
  * time      (time) object 108kB 1978-01-01 00:00:00 ... 2015-01-16 00:00:00
Data variables:
    FLDS      (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    FSDS      (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    PRECT     (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    PS        (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    QREFHT    (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    RHREFHT   (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHT    (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHTMN  (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHTMX  (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    U10       (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_...
    logname:           tilmes
    initial_file:      b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.001.cam.i....
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [18]:
combined_vds = combined_vds.drop_vars(["ilev", "lev"])

In [19]:
combined_vds

<xarray.Dataset> Size: 30GB
Dimensions:   (time: 13522, lat: 192, lon: 288)
Coordinates:
  * lat       (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * time      (time) object 108kB 1978-01-01 00:00:00 ... 2015-01-16 00:00:00
Data variables:
    FLDS      (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    FSDS      (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    PRECT     (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    PS        (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    QREFHT    (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    RHREFHT   (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHT    (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHTMN  (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    TREFHTMX  (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
    U10       (time, lat, lon) float32 3GB ManifestArray<shape=(13522, 192, 2...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_...
    logname:           tilmes
    initial_file:      b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.001.cam.i....
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [21]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        "s3://carbonplan-srm/",
        store=icechunk.s3_store(region="us-west-2"),
    ),
)


storage = icechunk.s3_storage(
    bucket="carbonplan-srm", prefix=virtual_ic_prefix, from_env=True
)
repo = icechunk.Repository.open_or_create(storage, config)
session = repo.writable_session("main")

In [20]:
virtual_ic_prefix

'input/tensor/CESM2-WACCM-Historical/icechunk/virtual_icechunk'

In [22]:
combined_vds.vz.to_icechunk(session.store)
snapshot_id = session.commit("virtual_CESM2-WACCM-Historical")
print(snapshot_id)
repo.save_config()

1SXGMBAT2GVB8AGMPB0G


In [23]:
credentials = icechunk.containers_credentials(
    {
        "s3://carbonplan-srm": icechunk.s3_credentials(),
    }
)

vz_repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)
vz_session = vz_repo.readonly_session("main")

In [24]:
ds = xr.open_zarr(
    vz_session.store,
    zarr_format=3,
    consolidated=False,
    chunks={},
)

ds

<xarray.Dataset> Size: 30GB
Dimensions:   (time: 13522, lat: 192, lon: 288)
Coordinates:
  * time      (time) object 108kB 1978-01-01 00:00:00 ... 2015-01-16 00:00:00
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * lat       (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
Data variables:
    TREFHTMN  (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    FLDS      (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    RHREFHT   (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    QREFHT    (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    PRECT     (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    TREFHT    (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    U10       (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    FSDS      (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    PS        (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    TREFHTMX  (time, lat, lon) float32 3GB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_...
    logname:           tilmes
    initial_file:      b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.001.cam.i....
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [43]:
write_storage_config = icechunk.s3_storage(bucket="carbonplan-srm", prefix=ic_prefix)
write_repo = icechunk.Repository.open_or_create(write_storage_config)
write_session = write_repo.writable_session("main")

In [44]:
ds = ds.chunk({"time": -1, "lat": 32, "lon": 48})
ds = ds.drop_encoding()

In [45]:
to_icechunk(ds, write_session)

In [46]:
first_snapshot = write_session.commit(
    "create spatialy chunked store: {'time':-1,'lat':32,'lon':48}"
)

In [47]:
first_snapshot

'7KWQW044XE4AYV0QK190'

In [48]:
rtds = xr.open_zarr(write_session.store)
rtds

<xarray.Dataset> Size: 30GB
Dimensions:   (time: 13522, lat: 192, lon: 288)
Coordinates:
  * time      (time) object 108kB 1978-01-01 00:00:00 ... 2015-01-16 00:00:00
  * lat       (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon       (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
Data variables:
    FLDS      (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    PRECT     (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    FSDS      (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    U10       (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    PS        (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    TREFHTMN  (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    QREFHT    (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    TREFHT    (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    TREFHTMX  (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
    RHREFHT   (time, lat, lon) float32 3GB dask.array<chunksize=(13522, 32, 48), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_...
    logname:           tilmes
    initial_file:      b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.001.cam.i....
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1